In [5]:
import os
from dotenv import load_dotenv

load_dotenv()

True

## Semantic search

In [6]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("resources/acmecorp-employee-handbook.pdf")

data = loader.load()

print(data)

incorrect startxref pointer(1)
parsing for Object Streams


[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-11-20T23:23:16+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2025-11-20T23:23:16+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'resources/acmecorp-employee-handbook.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Employee Handbook\nNon-Disclosure Agreement (NDA) Policy\nEmployees must protect confidential information belonging to the company, its clients, and partners.\nThis includes, but is not limited to, product roadmaps, customer data, internal communications,\nproprietary algorithms, financial information, and unreleased features. Confidential information may not\nbe shared with unauthorized individuals inside or outside the organization. These obligations continue\nafter employment ends.\nWorkplace Conduct Policy\nEmployees must maintain a respectful, professional environment

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(data)

print(len(all_splits))

3


Embedding Models: https://docs.langchain.com/oss/python/integrations/text_embedding

In [8]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    base_url=os.environ.get("OPENAI_BASE_URL"),
    api_key=os.environ.get("OPENAI_API_KEY"),
    model="bge-m3",
)

In [9]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

In [10]:
ids = vector_store.add_documents(documents=all_splits)

In [11]:
results = vector_store.similarity_search(
    "How many days of vacation does an employee get in their first year?"
)

print(results[0])

page_content='Employee Handbook
Non-Disclosure Agreement (NDA) Policy
Employees must protect confidential information belonging to the company, its clients, and partners.
This includes, but is not limited to, product roadmaps, customer data, internal communications,
proprietary algorithms, financial information, and unreleased features. Confidential information may not
be shared with unauthorized individuals inside or outside the organization. These obligations continue
after employment ends.
Workplace Conduct Policy
Employees must maintain a respectful, professional environment free from harassment, discrimination,
and intimidation. All employees are expected to follow organizational values, collaborate effectively,
and communicate constructively. Disruptive behavior, verbal abuse, or misuse of company systems is
prohibited. Violations may result in disciplinary action.
Paid Time Off (PTO) Policy
Full■time employees accrue PTO according to the following schedule:  0–1 years of servic

## RAG Agent

In [12]:
from langchain.tools import tool

@tool
def search_handbook(query: str) -> str:
    """Search the employee handbook for information"""
    results = vector_store.similarity_search(query)
    return results[0].page_content

In [13]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model_provider="openai",
    base_url=os.environ.get("OPENAI_BASE_URL"),
    api_key=os.environ.get("OPENAI_API_KEY"),
    model=os.environ.get("OPENAI_MODEL"),
    temperature=0.3,
)

agent = create_agent(
    model=model,
    tools=[search_handbook],
    system_prompt="You are a helpful agent that can search the employee handbook for information."
    )

In [14]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="How many days of vacation does an employee get in their first year?")]}
)

In [16]:
from rich import print as rprint

rprint(response)

{
    'messages': [
        HumanMessage(
            content='How many days of vacation does an employee get in their first year?',
            additional_kwargs={},
            response_metadata={},
            id='aaf5cef5-566d-47b4-bfae-d92d6738512c'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 30,
                    'prompt_tokens': 300,
                    'total_tokens': 330,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': None
                },
                'model_provider': 'openai',
                'model_name': 'Qwen3.5-397B-A17B-FP8',
                'system_fingerprint': None,
                'id': 'chatcmpl-b2bb74f6f08ff5a2',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019d295b-5ba7-70b0-aeb1-b837d04add8e-0',
            tool_calls=[
                {
                    'name': 'search_handbook',
                    'args': {'query': 'vacation days first year'},
                    'id': 'chatcmpl-tool-90beca31bfdc9aef',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 300,
                'output_tokens': 30,
                'total_tokens': 330,
                'input_token_details': {},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='business travel. This includes transportation, lodging, meals, and incidental expenses 
within\nestablished limits. Receipts must be submitted within 14 days of travel. First-class travel, 
personal\nexpenses, and non-business activities are not reimbursable. Employees should exercise good\njudgment and 
cost-effective decision-making when traveling on behalf of the company.',
            name='search_handbook',
            id='7c0f301a-f50f-48da-b2f7-e740d3b12d08',
            tool_call_id='chatcmpl-tool-90beca31bfdc9aef'
        ),
        AIMessage(
            content='The search results provided information regarding business travel reimbursement but did not 
contain details about vacation days for the first year.\n\nBased on general knowledge, vacation policies vary 
significantly by company. However, a common standard in many industries is **10 to 15 days** (or 2 to 3 weeks) of 
paid time off (PTO) for new employees in their first year. Some companies may offer less (e.g., 10 days) or accrue 
time based on hours worked.\n\nTo get the exact number for your specific situation, you would need to check your 
specific employment contract or the "Time Off" or "Benefits" section of your employee handbook directly, as the 
previous search did not locate that specific clause.',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 150,
                    'prompt_tokens': 418,
                    'total_tokens': 568,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': None
                },
                'model_provider': 'openai',
                'model_name': 'Qwen3.5-397B-A17B-FP8',
                'system_fingerprint': None,
                'id': 'chatcmpl-9eec8b8f7d40cf55',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--019d295b-5d7d-78a2-9225-75219997cdd7-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 418,
                'output_tokens': 150,
                'total_tokens': 568,
                'input_token_details': {},
                'output_token_details': {}
            }
        )
    ]
}